In [1]:
from firedrake import *
from firedrake.cython import dmcommon
import numpy as np

mesh = UnitSquareMesh(16, 16)
dm = mesh.topology_dm

tag = 17  # any integer marker you choose

# Make sure the facet label exists.
if not dm.hasLabel(dmcommon.FACE_SETS_LABEL):
    dm.createLabel(dmcommon.FACE_SETS_LABEL)

# In 2D, codim-1 facets are edges.
fstart, fend = dm.getHeightStratum(1)

coords = mesh.coordinates.dat.data_ro  # vertex coordinates

for f in range(fstart, fend):
    # Interior facets have two neighboring cells.
    if dm.getSupportSize(f) != 2:
        continue

    # Get facet vertices and compute a simple centroid.
    verts = dm.getCone(f)
    xmid = coords[np.array(verts)].mean(axis=0)

    # Mark the interior interface x = 0.5.
    if abs(xmid[0] - 0.5) < 1e-12:
        dm.setLabelValue(dmcommon.FACE_SETS_LABEL, f, tag)

V = FunctionSpace(mesh, "CG", 1)
u = TrialFunction(V)
v = TestFunction(V)

a = inner(grad(u), grad(v)) * dx
L = Constant(1.0) * v * dx

bc_interface = DirichletBC(V, Constant(2.0), tag)
uh = Function(V)

solve(a == L, uh, bcs=[bc_interface])

IndexError: index 513 is out of bounds for axis 0 with size 289

In [5]:
mesh = UnitSquareMesh(16, 16)
plex = mesh.topology_dm

tag = 17

# Build the coordinate-section mapping Firedrake uses for DMPlex points.
dim = mesh.topological_dimension()
gdim = mesh.geometric_dimension()
entity_dofs = np.zeros(dim + 1, dtype=np.int32)
entity_dofs[0] = gdim   # linear mesh: coordinates live on vertices
coord_section = mesh.create_section(entity_dofs)

coords = mesh.coordinates.dat.data_ro_with_halos

def point_coord(p):
    # For a vertex point p, get its coordinate tuple from the coordinate section.
    off = coord_section.getOffset(p) // gdim
    return coords[off:off + gdim]

# Ensure the facet label exists.
if not plex.hasLabel(dmcommon.FACE_SETS_LABEL):
    plex.createLabel(dmcommon.FACE_SETS_LABEL)

fstart, fend = plex.getHeightStratum(1)   # facets in 2D
vstart, vend = plex.getDepthStratum(0)    # vertices

for f in range(fstart, fend):
    if plex.getSupportSize(f) != 2:
        continue  # only interior facets

    # The cone contains points of lower dimension; keep just the vertices.
    verts = [p for p in plex.getCone(f) if vstart <= p < vend]
    if len(verts) != 2:
        continue

    xmid = 0.5 * (point_coord(verts[0]) + point_coord(verts[1]))

    if abs(xmid[0] - 0.5) < 1e-12:
        plex.setLabelValue(dmcommon.FACE_SETS_LABEL, f, tag)

V = FunctionSpace(mesh, "CG", 1)
u = TrialFunction(V)
v = TestFunction(V)
a = inner(grad(u), grad(v)) * dx
L = Constant(1.0) * v * dx

bc_interface = DirichletBC(V, Constant(2.0), tag)
uh = Function(V)
solve(a == L, uh, bcs=[bc_interface])

<function point_coord at 0x1393f8040>


AttributeError: 'tuple' object has no attribute 'getOffset'